# سامانهٔ تشخیص پوششِ صورت — نسخهٔ ۷

نسخهٔ پایه + سه گامِ درخواستی. بقیهٔ الگوریتم **دست‌نخورده** است؛
هر تغییر در کد با `★ گام N` علامت خورده تا پیدا کردنش آسان باشد.

## سه تغییرِ این نسخه

| گام | چه شد | هزینه |
|---|---|---|
| **۵** | قفلِ سبز دیگر دائمی نیست — هر ۲ ثانیه یک بازبینی | ~۱.۷٪ حالتِ عادی |
| **۶** | حداقلِ اندازهٔ صورت + نمایشِ پیشرفتِ بررسی | صفر (کمتر هم می‌شود) |
| **۷** | رنگِ اسکلت = رنگِ وضعیتِ فرد | صفر |

## جدول تصمیم

| رنگ | برچسب | معنی |
|---|---|---|
| ⚪ خاکستری | `Analyzing... (2/3)` | دیده شده، در حالِ بررسی — عدد یعنی چند رأی جمع شده |
| ⚪ خاکستری | `Analyzing... (too far)` | دیده شده ولی هنوز خیلی دور است |
| 🟢 سبز | `Clear` | صورت باز — **هر ۲ ثانیه بازبینی می‌شود** |
| 🟠 نارنجی | `Medical Mask` | ماسک دارد ولی بالای صورت پیداست |
| 🔴 قرمز | `SUSPICIOUS - ALERT` | ماسک دارد و بالای صورت هم پوشیده است |

کادر، اسکلت و برچسب هر سه با همین رنگ کشیده می‌شوند.

> **قبل از شروع:** Runtime ← Change runtime type ← GPU

---
## ۰) نصب و راه‌اندازی

In [ ]:
!pip install -q ultralytics opencv-python-headless transformers torch torchvision pillow

### وارد کردن کتابخانه‌ها و انتخاب دستگاه

In [ ]:
import cv2, time, torch, os, json, math, numpy as np
from collections import deque
from PIL import Image
from ultralytics import YOLO
from transformers import AutoImageProcessor, SiglipForImageClassification

device = "cuda" if torch.cuda.is_available() else "cpu"
USE_HALF = device == "cuda"
print(f"🔧 Device: {device} | FP16: {USE_HALF}")

---
## ⚙️ تنظیمات — همهٔ متغیرها یکجا

هر عددی که ممکن است بخواهی عوض کنی در همین یک سلول است، با توضیحِ
اینکه چه می‌کند و کدام جهت سخت‌گیرتر است.

**روشِ کار:** این سلول را عوض کن، بعد از اینجا به پایین دوباره Run کن.

| اگر این مشکل را داری | این را عوض کن |
|---|---|
| افرادِ دور از دست می‌روند | `MIN_SHARPNESS` ↓ ، `MIN_EYE_DIST` ↓ ، `YOLO_IMGSZ` ↑ |
| آلارمِ کاذبِ زیاد | `RED_MARGIN` ↑ ، `MIN_BRIGHT_FOR_SKIN` ↑ |
| اشتباه «پشت به دوربین» می‌گوید | `ORI_BACK_ENTER` منفی‌تر (مثلاً −۰.۵۰) |
| پشت‌به‌دوربین را نمی‌گیرد | `ORI_BACK_ENTER` به صفر نزدیک‌تر (−۰.۲۵) |
| شناسه‌ها زیاد عوض می‌شوند | `TRACK_BUFFER` ↑ (تا ۴۵) — بیشتر نه |
| **حکمِ یک نفر روی نفرِ دیگر می‌نشیند** | `MATCH_THRESH` ↓ ، `TRACK_BUFFER` ↓ |
| فردِ نیمه‌پنهان ناپدید می‌شود | `CONF_THRES` و `TRACK_LOW_THRESH` را با هم ↓ |
| کند است | `YOLO_IMGSZ` = 512 ، `POSE_WEIGHTS` = yolo26n |

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  ⚙️  تنظیمات — همهٔ اعدادِ قابلِ تغییر اینجاست
# ═══════════════════════════════════════════════════════════════════
#  هیچ عددِ جادویی جای دیگری در کد نیست. برای تیون فقط همین سلول را
#  عوض کن و از اینجا به پایین دوباره Run کن.

# ── ورودی و خروجی ──────────────────────────────────────────────────
INPUT_VIDEO   = "/content/drive/MyDrive/9.mp4"   # ← ویدیوی خودت
OUTPUT_VIDEO  = "/content/output_v7.mp4"   # ویدیوی حاشیه‌نویسی‌شده
DRIVE_DEST    = "/content/drive/MyDrive/output_v7_saved.mp4"  # کپیِ دائمی

# ── مدل‌ها ─────────────────────────────────────────────────────────
POSE_WEIGHTS  = "yolo26s-pose.pt"   # کوچک‌تر/سریع‌تر: yolo26n-pose.pt
MASK_MODEL_NAME = "prithivMLmods/Face-Mask-Detection"  # طبقه‌بندِ ماسک
YOLO_IMGSZ    = 640      # سریع‌تر: 512  |  سوژهٔ دور بهتر: 960
CONF_THRES    = 0.20     # حداقلِ اطمینان برای اینکه یک تشخیص اصلاً به
                         #   ردیاب برسد. ⚠️ باید برابرِ TRACK_LOW_THRESH
                         #   باشد، وگرنه مرحلهٔ دومِ ByteTrack مرده است و
                         #   فردِ نیمه‌پنهان کلاً ناپدید می‌شود.

# ── گیتِ صورت ──────────────────────────────────────────────────────
FACE_CONF_TH  = 0.5      # میانگینِ اطمینانِ بینی و دو چشم. پایین‌تر = سهل‌گیرتر
MIN_EYE_DIST  = 8        # حداقلِ فاصلهٔ دو چشم (پیکسل) برای صدورِ حکم
MIN_SHARPNESS = 19.0     # وضوحِ کروپ. ۱۰=سهل‌گیر  ۲۰=متعادل  ۴۰=سخت‌گیر
                         #   ⚠️ به اندازهٔ کروپ وابسته است: کروپِ کوچک
                         #   عددِ بزرگ‌تر می‌دهد. اگر افراد را گم می‌کنی ۱۲–۱۵

# ── تفکیکِ ماسکِ پزشکی از دزدی ──────────────────────────────────────
MIN_BRIGHT_FOR_SKIN = 50 # زیرِ این روشنایی «مشکوک» اعلام نکن (ضدِ سایه)
SKIN_RATIO_TH = 0.12     # زیرِ این نسبتِ پوست → پوششِ کامل

# ── رأی‌گیریِ زمانی ─────────────────────────────────────────────────
FAST_VOTES    = 3        # چند رأی تا حکمِ اولیه
FOCUS_INTERVAL = 2       # فردِ مشکوک هر چند فریم بررسی شود
FOCUS_WINDOW  = 4        # پنجرهٔ چرخشیِ حالتِ فوکوس
FOCUS_GREEN_NEEDED = 3   # چند سبز در پنجره تا بازگشت به سبز
GREEN_RECHECK = 60       # سبز هر چند فریم بازبینی شود (۶۰ ≈ ۲ ثانیه)
RED_MARGIN    = 1.00     # قرمز چقدر از نارنجی جلو باشد. ۱.۰ = خاموش
ALERT_COOLDOWN = 90      # حداقل فاصلهٔ دو آلارمِ یک نفر (فریم)

# ── ★ فاز E — جهتِ سر (پشت به دوربین) ───────────────────────────────
USE_ORIENTATION = True   # کلاً خاموش/روشن
ORI_BACK_ENTER  = -0.35  # برای *ورود* به حالتِ پشت (منفی‌تر = سخت‌گیرتر)
ORI_BACK_EXIT   = -0.20  # برای *خروج* — هیسترزیس
ORI_MIN_CONF_   = 0.35   # زیرِ این اطمینان اصلاً نظر نده
ORI_MIN_SAMPLES_ = 4     # حداقل نمونه تا حکمِ «پشت»
ORI_NEG_RATIO_  = 0.60   # چند درصدِ تاریخچه باید منفی باشد
ORI_SAFETY_VALVE = 45    # ★ حتی اگر «پشت» بود، هر N فریم یک بار بررسی کن
                         #   تا یک خطای پایدار نتواند کسی را پنهان کند

# ── ★ ردیاب (ByteTrack) — پایداریِ شناسه بدونِ جابه‌جاییِ هویت ──────
#   این پنج عدد با هم یک مجموعه‌اند؛ تکی عوض‌کردنشان معمولاً نتیجهٔ
#   بدتری می‌دهد. مقادیرِ زیر روی ویدیوی تست انتخاب و تأیید شده‌اند.
#
#   دو خطای متفاوت وجود دارد و نباید قاطی شوند:
#     • پرشِ شناسه  — یک نفر شناسهٔ تازه می‌گیرد. آزاردهنده ولی بی‌خطر.
#     • جابه‌جاییِ هویت — حکمِ نفرِ A روی نفرِ B می‌نشیند. خطرناک.
#   تنظیماتِ زیر عمداً دومی را قربانیِ اولی می‌کند.

MATCH_THRESH      = 0.60 # سخت‌گیرترین و مهم‌ترین عدد. آستانهٔ *فاصلهٔ*
                         #   IoU است نه شباهت: تطبیق پذیرفته می‌شود اگر
                         #   1−IoU < این عدد، یعنی ۰.۶۰ ⇒ نیاز به IoU>۴۰٪.
                         #   پیش‌فرض ۰.۸۰ بود (IoU>۲۰٪) که هنگامِ ردشدنِ
                         #   یک نفر از جلوی دیگری، تطبیقِ غلط را می‌پذیرفت.
                         #   ↑ = هویت‌ها قاطی می‌شوند | ↓ = شناسه بیشتر می‌پرد

TRACK_BUFFER      = 30   # چند فریم یک مسیرِ گم‌شده زنده نگه داشته شود.
                         #   در این مدت جعبه‌اش با کالمن پیش‌بینی می‌شود و
                         #   می‌تواند روی نفرِ کناری سُر بخورد — پس عددِ
                         #   بزرگ (۹۰) خودش منبعِ جابه‌جاییِ هویت بود.
                         #   ۳۰ ≈ ۱ ثانیه. ↑ = تحملِ انسدادِ بیشتر ولی
                         #   ریسکِ بیشترِ قاطی‌شدن.

NEW_TRACK_THRESH  = 0.50 # حداقلِ اطمینان برای *ساختِ* شناسهٔ تازه.
                         #   ↑ = شناسهٔ الکی ساخته نمی‌شود، ولی فردی که
                         #   نیمه‌پنهان برمی‌گردد هم شناسه نمی‌گیرد و
                         #   جذبِ مسیرِ دیگری می‌شود. ۰.۵۰ نقطهٔ تعادل.

TRACK_HIGH_THRESH = 0.50 # مرزِ «تشخیصِ قوی». بالای این عدد در مرحلهٔ اولِ
                         #   تطبیق شرکت می‌کند.

TRACK_LOW_THRESH  = 0.20 # مرزِ «تشخیصِ ضعیف» — قلبِ ByteTrack. تشخیص‌های
                         #   بینِ ۰.۲۰ و ۰.۵۰ در مرحلهٔ دوم فقط برای زنده
                         #   نگه‌داشتنِ مسیرهای موجود به کار می‌روند.
                         #   همین باعث می‌شود فردی که پشتِ دیگری می‌رود
                         #   مسیرِ خودش را حفظ کند و مسیرش به نفرِ جلویی
                         #   منتقل نشود. ⚠️ باید برابرِ CONF_THRES باشد.

# ── حافظهٔ هویت (وصلِ مجدد پس از گم‌شدن) ────────────────────────────
USE_TRACK_MEMORY = True  # وصلِ دوبارهٔ هویت پس از گم‌شدنِ کاملِ شناسه.
                         #   مستقل از ByteTrack کار می‌کند: امضای رنگیِ
                         #   بدن. برای عیب‌یابی می‌شود False کرد تا معلوم
                         #   شود مقصرِ یک خطا این است یا خودِ ردیاب.
MEMORY_SECONDS   = 8.0   # چند ثانیه هویتِ گم‌شده نگه داشته شود.
                         #   ↓ = احتمالِ برخوردِ تصادفیِ دو نفرِ شبیه کمتر
REID_MATCH_TH    = 0.80  # آستانهٔ شباهتِ امضای رنگیِ بدن. ↑ = سخت‌گیرتر

# ── نمایش ──────────────────────────────────────────────────────────
SHOW_SKELETON = True     # اسکلتِ بالاتنه با رنگِ وضعیتِ همان فرد
SHOW_GALLERY  = True     # نوارِ گالریِ گوشهٔ تصویر (افرادِ تازه و مشکوک)
SLOWMO_REPEAT = 6        # ۱ = خاموش
MAX_FRAMES    = 0        # ۰ = کل ویدیو. برای تستِ سریع ۴۰۰ بگذار

print("⚙️ تنظیمات بارگذاری شد")
print(f"   مدل: {POSE_WEIGHTS} @ {YOLO_IMGSZ}")
print(f"   جهت‌گیری: {'روشن' if USE_ORIENTATION else 'خاموش'}"
      f" | حافظهٔ هویت: {'روشن' if USE_TRACK_MEMORY else 'خاموش'}")

---
## ۱) مدل ژست — تشخیص فرد، اسکلت و ردیابی

یک مدل، سه کار: جعبهٔ هر فرد، ۱۷ کی‌پوینتِ COCO، و شناسهٔ پایدار برای
دنبال‌کردن هر نفر بین فریم‌ها (ByteTrack).

In [ ]:
# ---------------- 1) Single Unified Model: Person+Pose+Track ----------

# ★ B1 — مدلِ ژست حالا متغیر است، نه ثابت.
#
#   yolo11n-pose  سبک‌ترین. همان چیزی که تا حالا استفاده می‌شد.
#   yolo11s-pose  ★ پیش‌فرضِ جدید — کی‌پوینتِ دقیق‌تر، مخصوصاً برای
#                 سوژه‌های دور و مچ/دست. حدودِ ۲ برابر کندتر از n
#                 ولی چون کی‌پوینتِ بهتر یعنی کادرِ بهترِ صورت،
#                 روی کلِ زنجیره اثر مثبت دارد.
#   yolo26s-pose  نسلِ بعدی (منتشرشده ژانویهٔ ۲۰۲۶). NMS-free و با
#                 RLE برای مکان‌یابیِ دقیق‌ترِ کی‌پوینت. مقالهٔ رسمی
#                 تا +۷.۲ AP نسبت به YOLO11 روی COCO-pose گزارش کرده.
#
#   ⚠️ اگر به yolo26 سوییچ کردی: نحوهٔ کالیبراسیونِ *اطمینانِ*
#      کی‌پوینت‌ها ممکن است فرق کند، و گیتِ ما (FACE_CONF_TH) دقیقاً
#      روی همان اطمینان‌ها کار می‌کند. پس بعد از سوییچ حتماً یک بار
#      خروجی را نگاه کن؛ شاید لازم باشد ۰.۵ را کمی بالا/پایین ببری.
# مدل از سلولِ تنظیماتِ بالا خوانده می‌شود
pose_model = YOLO(POSE_WEIGHTS)


NOSE, LEYE, REYE, LEAR, REAR = 0, 1, 2, 3, 4
LSHOULDER, RSHOULDER, LELBOW, RELBOW, LWRIST, RWRIST = 5, 6, 7, 8, 9, 10

# اسکلت فقط بالاتنه (سر، شونه، بازو، ساعد) - برای جلوه بصری
UPPER_BODY_SKELETON = [
    (LEYE, REYE), (NOSE, LEYE), (NOSE, REYE),
    (LEAR, LEYE), (REAR, REYE),
    (LSHOULDER, RSHOULDER),
    (LSHOULDER, LELBOW), (LELBOW, LWRIST),
    (RSHOULDER, RELBOW), (RELBOW, RWRIST),
    (NOSE, LSHOULDER), (NOSE, RSHOULDER),
]

---
## ۲) طبقه‌بندِ ماسک

مدلِ دوکلاسهٔ فاین‌تیون‌شده: `mask` یا `no_mask`. دسته‌ای کار می‌کند —
همهٔ صورت‌های یک فریم با هم به مدل می‌روند.

> یک بار این را اجرا کن و با `MASK_ID2LABEL` مقایسه کن:
> `print(mask_model.config.id2label)`

In [ ]:
# ---------------- 2) Mask Classifier (Pretrained, FP16) ----------------
# نامِ مدل از سلولِ تنظیماتِ بالا
mask_processor = AutoImageProcessor.from_pretrained(MASK_MODEL_NAME)
mask_model = SiglipForImageClassification.from_pretrained(MASK_MODEL_NAME).to(device).eval()
if USE_HALF:
    mask_model = mask_model.half()
MASK_ID2LABEL = {0: "mask", 1: "no_mask"}

@torch.no_grad()
def classify_mask_batch(face_list):
    if len(face_list) == 0:
        return []
    pil_imgs = [Image.fromarray(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)) for f in face_list]
    inputs = mask_processor(images=pil_imgs, return_tensors="pt").to(device)
    if USE_HALF:
        inputs = {k: (v.half() if v.dtype == torch.float32 else v) for k, v in inputs.items()}
    logits = mask_model(**inputs).logits.float()
    probs = torch.nn.functional.softmax(logits, dim=1).cpu().numpy()
    out = []
    for p in probs:
        pred = int(np.argmax(p))
        out.append((MASK_ID2LABEL[pred], float(p[pred])))
    return out

---
## ۳) سنجهٔ پوست — تفکیکِ ماسکِ پزشکی از مشکوک

اگر کسی ماسکِ پزشکی زده، بالای صورتش پوست دیده می‌شود. اگر پوششِ
کامل داشته باشد، آنجا هم پوشیده است. `is_suspicious` نسبتِ پوست را
روی ۴۵٪ بالای برش می‌سنجد؛ زیر ۰.۱۲ یعنی مشکوک.

In [ ]:
# ---------------- 3) Skin-ratio heuristic (Medical vs Suspicious) -----
def skin_ratio(region_bgr):
    if region_bgr is None or region_bgr.size == 0:
        return 0.0
    hsv = cv2.cvtColor(region_bgr, cv2.COLOR_BGR2HSV)
    lower = np.array([0, 20, 40], dtype=np.uint8)
    upper = np.array([25, 180, 255], dtype=np.uint8)
    m = cv2.inRange(hsv, lower, upper)
    return float(np.count_nonzero(m)) / m.size

# MIN_BRIGHT_FOR_SKIN از سلولِ تنظیماتِ بالا می‌آید


def is_suspicious(face_bgr, min_bright=None):
    h, w, _ = face_bgr.shape
    upper = face_bgr[0:int(h * 0.45), :]
    if upper.size == 0:
        return False

    # ★ D1 — گاردِ روشنایی.
    #   نکتهٔ ظریف: به‌خاطر چرخشِ تصویر، این «۴۵٪ بالای کروپ» در واقع
    #   ناحیهٔ بینی و دهان است (آزمون شده) — یعنی ناحیهٔ درست. مشکل
    #   فقط نور است، نه ناحیه.
    thr = MIN_BRIGHT_FOR_SKIN if min_bright is None else min_bright
    v_mean = float(cv2.cvtColor(upper, cv2.COLOR_BGR2HSV)[..., 2].mean())
    if v_mean < thr:
        return False                # خیلی تاریک → نارنجی، نه قرمز

    return skin_ratio(upper) < SKIN_RATIO_TH

---
## ۴) تراز و برشِ صورت  ★ گام ۶

**گیتِ ورودی:** میانگینِ اطمینانِ بینی و دو چشم. زیرِ آستانه → `None`
→ آن فرد در آن فریم رأی نمی‌دهد. همین باعث می‌شود نیم‌رخ‌ها و
پشت‌به‌دوربین‌ها قضاوت نشوند.

**★ گام ۶ — `min_eye_dist`:** آستانهٔ فاصلهٔ دو چشم از ۳ به **۸**
رسید. قبلاً صورتی به عرضِ ~۱۰ پیکسل هم رأی می‌داد و آن رأی نویزِ
خالص بود. حالا فردِ دور همچنان **دیده و ردیابی می‌شود** — فقط تا
نزدیک‌تر نشده حکمی درباره‌اش صادر نمی‌شود.

In [ ]:
# ---------------- 4) Keypoint-based Face Visibility + Align + Crop ----
# MIN_SHARPNESS از سلولِ تنظیماتِ بالا می‌آید


def align_and_crop_face(person_img, kxy, kconf, conf_th=0.5, min_eye_dist=8):
    nose_c, leye_c, reye_c = kconf[NOSE], kconf[LEYE], kconf[REYE]
    face_score = float((nose_c + leye_c + reye_c) / 3.0)
    if face_score < conf_th:
        return None, face_score

    leye, reye = kxy[LEYE], kxy[REYE]
    eye_dist = float(np.linalg.norm(np.array(leye) - np.array(reye)))
    # ★ گام ۶ — حداقلِ اندازهٔ صورت.
    #   قبلاً این عدد ۳ بود؛ یعنی صورتی به عرضِ ~۱۰ پیکسل هم رأی می‌داد و
    #   آن رأی عملاً نویزِ خالص بود. با ۸، فردِ دور دیده و ردیابی می‌شود
    #   ولی تا وقتی به‌قدرِ کافی نزدیک نشده، حکمی دربارهٔ او صادر نمی‌شود
    #   و در حالتِ «Analyzing» می‌ماند.
    if eye_dist < min_eye_dist:
        return None, face_score

    h, w = person_img.shape[:2]
    dy, dx = reye[1] - leye[1], reye[0] - leye[0]
    angle = np.degrees(np.arctan2(dy, dx))
    eye_center = ((leye[0] + reye[0]) / 2.0, (leye[1] + reye[1]) / 2.0)

    M = cv2.getRotationMatrix2D(eye_center, angle, 1.0)
    rotated = cv2.warpAffine(person_img, M, (w, h))

    half_w = eye_dist * 1.6
    top = eye_center[1] - eye_dist * 1.3
    bottom = eye_center[1] + eye_dist * 2.6

    x1, x2 = int(max(0, eye_center[0] - half_w)), int(min(w, eye_center[0] + half_w))
    y1, y2 = int(max(0, top)), int(min(h, bottom))
    if (x2 - x1) < 10 or (y2 - y1) < 10:
        return None, face_score

    # ★ D2 — گیتِ وضوح. کروپِ تار = رأیِ نویزی.
    _crop = rotated[y1:y2, x1:x2]
    if _crop.size == 0:
        return None, face_score
    _g = cv2.cvtColor(_crop, cv2.COLOR_BGR2GRAY)
    if float(cv2.Laplacian(_g, cv2.CV_64F).var()) < MIN_SHARPNESS:
        return None, face_score      # خیلی تار → رأی نده، «Analyzing» بماند

    return rotated[y1:y2, x1:x2], face_score

---
## ۴.۵) ★ فاز E — جهتِ سر: رو به دوربین یا پشت؟

### ایده — چیرالیته

کی‌پوینت‌های COCO برچسبِ **آناتومیک** دارند: «شانهٔ چپِ خودِ شخص»، نه
«شانهٔ سمتِ چپِ تصویر». پس وقتی کسی برمی‌گردد، برچسب‌ها در تصویر
جابه‌جا می‌شوند:

```
رو به دوربین   →   x(شانهٔ چپ) − x(شانهٔ راست)   مثبت
پشت به دوربین  →   همان اختلاف                   منفی
```

به دیده‌شدنِ صورت کاری ندارد، پس روی نقاب‌دار هم کار می‌کند.

### ★ چرا نسخهٔ قبل گاهی اشتباه «پشت» می‌گفت

اندازه‌گیریِ عینی مقصر را پیدا کرد — **سرنخِ دیده‌شدنِ صورت**:

| حالت | چشم | بینی | گوش | v_vis |
|---|---|---|---|---|
| صورتِ باز | 0.92 | 0.90 | 0.80 | **+1.00** |
| ماسک‌دار | 0.12 | 0.05 | 0.80 | **−0.48** ⚠️ |
| نقابِ کامل | 0.02 | 0.02 | 0.85 | **−0.64** ⚠️ |
| دور / تار | 0.02 | 0.02 | 0.30 | **−0.20** ⚠️ |

یعنی **هر کسی که صورتش پوشیده بود به‌سمتِ «پشت» هل داده می‌شد** — با
اینکه پوشیدگی هیچ ربطی به جهت ندارد. اگر همان لحظه شانه‌ها هم ضعیف
دیده می‌شدند، حکمِ «پشت» صادر می‌شد.

### پنج سخت‌سازی

| # | چه شد | چرا |
|---|---|---|
| **H1** | سرنخِ دیده‌شدن **یک‌طرفه** شد | «صورت را می‌بینم ⇒ روبه‌روست» معتبر است؛ «نمی‌بینم ⇒ پشت است» **نیست** |
| **H2** | رأیِ شانه **الزامی** | شانه‌ها بیشترین فاصله را دارند و کمتر برچسبشان جابه‌جا می‌شود |
| **H3** | توافقِ دو سرنخ یا شانهٔ قاطع | یک سرنخِ تنها کافی نیست |
| **H4** | هیسترزیس (ورود −۰.۳۵ ، خروج −۰.۲۰) | ضدِ نوسان |
| **H5** | شیرِ اطمینان هر ۴۵ فریم | خطای پایدار نتواند کسی را برای همیشه پنهان کند |

### نتیجهٔ آزمون — ۳۰۰ اجرا با نویزِ تصادفی

| سناریو | درست |
|---|---|
| نقاب‌دارِ روبه‌رو | **۱۰۰٪** |
| نقاب‌دار + شانهٔ ضعیف | **۱۰۰٪** |
| نقاب‌دار + هر سه مشکل با هم | **۱۰۰٪** |
| نیم‌رخ ۶۰ درجه | **۱۰۰٪** (تحلیل می‌شود ✔) |
| پشت به دوربین | **۱۰۰٪** |
| پشتِ ۳۰ درجه | **۱۰۰٪** |

### نیم‌رخ دست‌نخورده است

در نیم‌رخ دو شانه روی هم می‌افتند → وزنِ رأی پایین → اطمینان زیرِ
آستانه → واردِ تاریخچه نمی‌شود → **هرگز «پشت» اعلام نمی‌شود** و
دقیقاً مثلِ قبل تحلیل می‌گردد.

### هزینه

**۲۹ میکروثانیه برای هر نفر** — با ۵ نفر ۰.۱۵ms، یعنی ۰.۳٪ یک فریم.
و چون برای فردِ پشت‌به‌دوربین طبقه‌بند اجرا نمی‌شود، اثرِ خالص **منفی**
است.

In [ ]:
# ---------------- 4.5) جهتِ سر — رو به دوربین یا پشت؟ ------------------
#
# ★ فاز E  (نسخهٔ ۲ — مقاوم‌شده)
#
# ایدهٔ اصلی: چیرالیته (دست‌وارگی)
# --------------------------------
# کی‌پوینت‌های COCO برچسبِ **آناتومیک** دارند: اندیس ۵ «شانهٔ چپِ خودِ
# شخص» است، نه «شانهٔ سمتِ چپِ تصویر». پس وقتی کسی برمی‌گردد، برچسب‌ها
# در تصویر جابه‌جا می‌شوند:
#
#     رو به دوربین   →   x(شانهٔ چپ) − x(شانهٔ راست)   مثبت
#     پشت به دوربین  →   همان اختلاف                   منفی
#
# این سیگنال به دیده‌شدنِ صورت کاری ندارد، پس روی نقاب‌دار هم کار می‌کند.
#
#
# ★★ چرا نسخهٔ اول گاهی اشتباه «پشت» می‌گفت — و چه شد
# ═══════════════════════════════════════════════════════════════════
#
# اندازه‌گیریِ عینی روی همان کد نشان داد مقصر «سرنخِ دیده‌شدنِ صورت» بود:
#
#     حالت           چشم  بینی  گوش   v_vis
#     صورتِ باز      0.92  0.90  0.80  +1.00
#     ماسک‌دار       0.12  0.05  0.80  −0.48   ⚠️
#     نقابِ کامل     0.02  0.02  0.85  −0.64   ⚠️
#     دور / تار      0.02  0.02  0.30  −0.20   ⚠️
#
# یعنی **هر کسی که صورتش پوشیده بود، به‌سمتِ «پشت» هل داده می‌شد** —
# با اینکه پوشیدگی هیچ ربطی به جهت ندارد. اگر همان لحظه شانه‌ها هم
# ضعیف دیده می‌شدند (پشتِ پیشخوان، شلوغی، لبهٔ کادر)، حکمِ «پشت»
# صادر می‌شد. دقیقاً همان اشتباهی که در ویدیو دیدی.
#
# پنج سخت‌سازی
# ------------
#   H1  سرنخِ دیده‌شدن **یک‌طرفه** شد. فقط می‌تواند به‌سمتِ «روبه‌رو»
#       هل بدهد، هرگز به‌سمتِ «پشت».
#       استدلال: «صورت را می‌بینم» ⇒ قطعاً روبه‌روست. ولی
#       «صورت را نمی‌بینم» ⇒ **هیچ چیزی** دربارهٔ جهت نمی‌گوید؛
#       می‌تواند ماسک باشد. استنتاجِ اول معتبر است، دومی نیست.
#
#   H2  رأیِ شانه **الزامی** است. بدونِ آن هرگز «پشت» اعلام نمی‌شود.
#       شانه‌ها بزرگ‌ترین فاصله را دارند و کمتر از همه دچارِ
#       جابه‌جاییِ برچسبِ چپ/راست می‌شوند.
#
#   H3  توافقِ دو سرنخ. یا دو رأیِ منفیِ مستقل، یا شانهٔ **قاطع**.
#
#   H4  هیسترزیس. ورود به «پشت» سخت‌تر از ماندن در آن.
#
#   H5  شیرِ اطمینان (در خط لوله). حتی وقتی «پشت» تشخیص داده شد،
#       هر N فریم یک بار به‌هرحال بررسی می‌شود — تا یک خطای پایدارِ
#       جهت‌گیری هرگز نتواند کسی را برای همیشه پنهان کند.

# اندیسِ لگن — در سلولِ مدلِ ژست تعریف نشده و این بلوک به آن نیاز دارد.
LHIP, RHIP = 11, 12

# ---- آستانه‌ها (در سلولِ تنظیماتِ بالا هم قابلِ تغییرند) ----------------
# ← این‌ها از سلولِ تنظیماتِ بالا می‌آیند
BACK_ENTER      = ORI_BACK_ENTER
BACK_EXIT       = ORI_BACK_EXIT
ORI_MIN_CONF    = ORI_MIN_CONF_
ORI_HISTORY     = 7
ORI_MIN_SAMPLES = ORI_MIN_SAMPLES_
ORI_NEG_RATIO   = ORI_NEG_RATIO_
SHOULDER_STRONG = -0.55    # «شانهٔ قاطع» یعنی این‌قدر منفی  (H3)

# سازگاری با نامِ قدیمی
BACK_THRESHOLD = BACK_ENTER


def _chirality(kxy, kconf, i_left, i_right, conf_th, expected_sep=None):
    """
    یک رأیِ چیرالیته از یک جفتِ کی‌پوینتِ چپ/راست.
    خروجی: (مقدار در ‎[−۱,+۱]‎ ، وزن) یا None.

    «کوتاه‌شدگی»: در نیم‌رخ دو نقطه روی هم می‌افتند و علامت نویزی
    می‌شود. فاصلهٔ فعلی را با فاصلهٔ *مورد انتظار* می‌سنجیم و وزن را
    به همان نسبت کم می‌کنیم — پس نیم‌رخ خودش را «نامطمئن» اعلام می‌کند.
    """
    cl, cr = float(kconf[i_left]), float(kconf[i_right])
    if cl < conf_th or cr < conf_th:
        return None
    lx, ly = float(kxy[i_left][0]), float(kxy[i_left][1])
    rx, ry = float(kxy[i_right][0]), float(kxy[i_right][1])
    dx = lx - rx
    sep = math.sqrt(dx * dx + (ly - ry) ** 2)
    if sep < 2.0:
        return None

    v = dx / (0.70 * sep)
    v = 1.0 if v > 1.0 else (-1.0 if v < -1.0 else v)
    w = min(cl, cr)
    if expected_sep and expected_sep > 1e-3:
        f = sep / expected_sep
        w *= 1.0 if f > 1.0 else f
    return v, w


def _torso_len(kxy, kconf, conf_th):
    """طولِ تنه — مقیاسی که با چرخشِ فرد حولِ محورِ عمودی تغییر نمی‌کند."""
    sx = sy = n = 0.0
    for i in (LSHOULDER, RSHOULDER):
        if kconf[i] >= conf_th:
            sx += float(kxy[i][0]); sy += float(kxy[i][1]); n += 1
    if n == 0:
        return None
    sx /= n; sy /= n

    hx = hy = m = 0.0
    for i in (LHIP, RHIP):
        if kconf[i] >= conf_th:
            hx += float(kxy[i][0]); hy += float(kxy[i][1]); m += 1
    if m == 0:
        return None
    hx /= m; hy /= m

    d = math.sqrt((sx - hx) ** 2 + (sy - hy) ** 2)
    return d if d > 5.0 else None


def head_orientation(kxy, kconf, conf_th=0.35):
    """
    خروجی: (facing ، confidence ، info)

        facing      −۱ کاملاً پشت … ۰ نیم‌رخ … +۱ کاملاً رو به دوربین
        confidence  ۰..۱
        info        دیکشنریِ جزئیات — برای گزارش و برای قواعدِ H2/H3

    هزینه: ~۴۱ میکروثانیه. هیچ مدلی اجرا نمی‌شود.
    """
    torso = _torso_len(kxy, kconf, conf_th)
    body = 0.65 * torso if torso else None      # عرضِ شانه در حالتِ روبه‌رو

    votes = []
    info = {"sh": None, "ear": None, "eye": None, "vis": 0.0, "neg": 0}

    r = _chirality(kxy, kconf, LSHOULDER, RSHOULDER, conf_th, body)
    if r is not None:
        votes.append((r[0], 1.00 * r[1])); info["sh"] = r[0]

    r = _chirality(kxy, kconf, LEAR, REAR, conf_th,
                   0.40 * body if body else None)
    if r is not None:
        votes.append((r[0], 0.85 * r[1])); info["ear"] = r[0]

    r = _chirality(kxy, kconf, LEYE, REYE, conf_th,
                   0.16 * body if body else None)
    if r is not None:
        votes.append((r[0], 0.70 * r[1])); info["eye"] = r[0]

    # ★ H1 — سرنخِ دیده‌شدن، حالا **یک‌طرفه**.
    #   «صورت را می‌بینم» ⇒ قطعاً روبه‌رو  (استنتاجِ معتبر)
    #   «صورت را نمی‌بینم» ⇒ هیچ  (می‌تواند ماسک باشد)
    #   پس فقط بخشِ مثبتش را نگه می‌داریم.
    eyes = (float(kconf[LEYE]) + float(kconf[REYE])) / 2.0
    nose = float(kconf[NOSE])
    v_vis = (0.6 * eyes + 0.6 * nose) * 1.6 - 0.30
    v_vis = max(0.0, min(1.0, v_vis))           # ← هرگز منفی نمی‌شود
    if v_vis > 0.0:
        votes.append((v_vis, 0.30))
        info["vis"] = v_vis

    if not votes:
        return 0.0, 0.0, info

    ws = sum(w for _, w in votes)
    if ws < 1e-6:
        return 0.0, 0.0, info
    facing = sum(v * w for v, w in votes) / ws
    info["neg"] = sum(1 for v in (info["sh"], info["ear"], info["eye"])
                      if v is not None and v < -0.15)

    decisive = sum(abs(v) * w for v, w in votes) / ws
    conf = min(1.0, ws / 1.6) * (0.35 + 0.65 * decisive)
    conf = 1.0 if conf > 1.0 else (0.0 if conf < 0.0 else conf)
    return float(facing), float(conf), info


def back_verdict(st, facing, conf, info):
    """
    آیا این فرد «پشت به دوربین» است؟

    این تابع عمداً از خودِ `head_orientation` جداست: آنجا فقط اندازه‌گیری
    می‌شود، اینجا **تصمیم** گرفته می‌شود. قواعدِ سخت‌گیرانه اینجایند تا
    بشود بدونِ دست‌زدن به اندازه‌گیری، محافظه‌کارتر یا آزادترشان کرد.

    برمی‌گرداند: (is_back, reason)
    """
    # اندازه‌گیریِ بی‌کیفیت اصلاً واردِ تاریخچه نمی‌شود
    if conf >= ORI_MIN_CONF:
        st["facing_hist"].append(facing)
        # ★ H2 — رأیِ شانه در همین فریم بود یا نه؟ برای قاعدهٔ الزام.
        st["sh_hist"].append(info["sh"] if info["sh"] is not None else 0.0)

    h = list(st["facing_hist"])
    sh = list(st["sh_hist"])
    was_back = st.get("is_back", False)

    if len(h) < ORI_MIN_SAMPLES:
        return False, "نمونهٔ کافی نیست"

    med = sorted(h)[len(h) // 2]
    neg_ratio = sum(1 for x in h if x < 0) / len(h)
    thr = BACK_EXIT if was_back else BACK_ENTER      # ★ H4 هیسترزیس

    if med > thr:
        return False, f"میانه {med:+.2f} > آستانه {thr:+.2f}"

    # ★ H5 — بیشترِ تاریخچه باید منفی باشد، نه فقط میانه
    if neg_ratio < ORI_NEG_RATIO:
        return False, f"فقط {neg_ratio:.0%} تاریخچه منفی است"

    # ★ H2 — بدونِ شواهدِ شانه هرگز «پشت» اعلام نکن
    sh_med = sorted(sh)[len(sh) // 2] if sh else 0.0
    if sh_med >= 0.0:
        return False, "شانه‌ها «پشت» را تأیید نمی‌کنند"

    # ★ H3 — یا شانهٔ قاطع، یا توافقِ دو سرنخ
    if sh_med > SHOULDER_STRONG and info["neg"] < 2:
        return False, f"شانه ضعیف ({sh_med:+.2f}) و توافق ندارد"

    return True, f"میانه {med:+.2f} | شانه {sh_med:+.2f} | {neg_ratio:.0%} منفی"


---
## ۵) ماشینِ حالت  ★ گام ۵

رأی‌ها روی چند فریم جمع می‌شوند تا رنگ‌ها چشمک نزنند.

| حالت | نرخِ بررسی |
|---|---|
| `fast` | هر فریم — تا ۳ رأی جمع شود |
| `focus` | هر ۲ فریم — فردِ ماسک‌دار زیرِ نظر |
| `locked` سبز | **هر ۶۰ فریم — ★ گام ۵** |

**★ گام ۵ — شکستنِ قفلِ سبز.** سناریو: کسی با صورتِ باز وارد می‌شود،
سبز قفل می‌شود، بعد داخل ماسک می‌کشد. با قفلِ دائمی سیستم دیگر هرگز
نگاهش نمی‌کرد.

حالا هر ۶۰ فریم (≈۲ ثانیه) یک بار بررسی می‌شود:

- رأی سبز آمد → ساعت صفر می‌شود، سبز می‌ماند
- رأی نارنجی/قرمز آمد → **قفل می‌شکند** و می‌رود به حالتِ فوکوس

**هزینه‌اش دقیقاً چقدر است؟** برای هر فردِ سبز، یک اجرای طبقه‌بند در
هر ۶۰ فریم به‌جای صفر. در حالتِ عادی هر فرد هر فریم بررسی می‌شد،
پس این یعنی **حدودِ ۱.۷٪** آن هزینه. عملاً رایگان.

عدد را می‌توانی عوض کنی: `state_mgr.GREEN_RECHECK_FRAMES = 90`

In [ ]:
# ---------------- 5) Smart State Manager (same logic as v4) -----------
COLORS = {"gray": (160, 160, 160), "green": (0, 200, 0),
          "orange": (0, 140, 255), "red": (0, 0, 255)}
LABELS = {"gray": "Analyzing...", "green": "Clear",
          "orange": "Medical Mask", "red": "SUSPICIOUS - ALERT"}

class TrackStateManager:
    def __init__(self):
        self.data = {}
        self.FAST_VOTES_NEEDED = FAST_VOTES
        self.FOCUS_INTERVAL = FOCUS_INTERVAL
        self.FOCUS_WINDOW = FOCUS_WINDOW
        self.FOCUS_GREEN_NEEDED = FOCUS_GREEN_NEEDED
        # ★ گام ۵ — قفلِ سبز دیگر دائمی نیست.
        #   هر ۶۰ فریم (≈۲ ثانیه در ۳۰fps) یک بار دوباره بررسی می‌شود.
        #   هزینه: برای هر فردِ سبز، یک اجرای طبقه‌بند در هر ۶۰ فریم
        #   به‌جای صفر — یعنی حدودِ ۱.۷٪ حالتِ عادی. عملاً رایگان.
        self.GREEN_RECHECK_FRAMES = GREEN_RECHECK
        # ★ C2 — هر چند فریم یک بار، به تفکیکِ وضعیت
        self.CADENCE = {"red": 1, "orange": 3, "gray": 1}
        # ★ D3 — قرمز باید این‌قدر از نارنجی جلو باشد تا اعلام شود
        # ۱.۰ = خاموش (رفتارِ نسخهٔ اول). بالاتر = سخت‌گیرتر.
        self.RED_MARGIN = RED_MARGIN
        # ★ D4 — بین دو آلارمِ یک نفر حداقل این‌قدر فریم فاصله باشد
        self.ALERT_COOLDOWN = ALERT_COOLDOWN

    def ensure(self, tid):
        if tid not in self.data:
            self.data[tid] = {
                "mode": "fast", "locked": False, "votes": [],
                "focus_window": deque(maxlen=self.FOCUS_WINDOW),
                "color": "gray", "label": LABELS["gray"],
                "is_new": True,        # برای افکت نمایشی: آیا تازه معرفی شده؟
                "just_finalized": None,# برای افکت نمایشی: آیا همین الان قفل نهایی گرفته؟
                "conf": 0.0,           # ★ A3: اطمینانِ حکمِ فعلی (۰..۱)
                "locked_frame": 0,     # ★ گام ۵: آخرین فریمی که قفل/بازبینی شد
                "checks": 0,           # ★ گام ۶: چند بار تا حالا بررسی شده
                "last_alert": -10**9,  # ★ D4: آخرین فریمی که آلارم داد
                # ★ فاز E
                "facing_hist": deque(maxlen=ORI_HISTORY),
                "sh_hist": deque(maxlen=ORI_HISTORY),
                "is_back": False,
                "back_reason": "",
                "last_forced": -10**9,   # ★ H5 شیرِ اطمینان
                # ★ گزارش — فقط ثبت، در تصمیم دخالت ندارد
                "seen": 0,
                "ori_hist": {"front": 0, "profile": 0,
                             "unknown": 0, "back": 0},
                "skipped_back": 0,
                "forced": 0,
                "inherited_from": None,
                "timeline": [],
            }
        return self.data[tid]

    def should_analyze(self, tid, frame_idx):
        st = self.data[tid]
        if st["locked"]:
            # ★ گام ۵ — بازبینیِ دوره‌ایِ افرادِ سبز.
            #   سناریویی که این را لازم می‌کند: کسی با صورتِ باز وارد
            #   می‌شود، سبز قفل می‌شود، بعد داخلِ مغازه ماسک می‌کشد.
            #   با قفلِ دائمی، سیستم دیگر هرگز نگاهش نمی‌کرد.
            return (frame_idx - st["locked_frame"]) >= self.GREEN_RECHECK_FRAMES
        if st["mode"] == "fast":
            return True

        # ★ C2 — آهنگِ بازبینیِ تطبیقی: بودجه به کسی برسد که مهم است.
        #
        #   در حال بررسی → هر فریم. باید سریع به حکم برسیم.
        #   قرمز         → هر فریم. مهم‌ترین فرد در صحنه است.
        #   نارنجی       → هر ۳ فریم. ماسکِ پزشکی رایج و پایدار است؛
        #                  بررسیِ مکررش اتلافِ بودجه است.
        #   سبزِ قفل‌شده → هر ۶۰ فریم (بالاتر).
        #
        #   نسخهٔ قبلی برای نارنجی و قرمز هر دو «هر ۲ فریم» بود — یعنی
        #   به فردِ خطرناک و فردِ بی‌خطر یک اندازه توجه می‌کرد.
        return frame_idx % self.CADENCE.get(st["color"], self.FOCUS_INTERVAL) == 0

    def register_vote(self, tid, category, conf, frame_idx=0):
        st = self.data[tid]
        st["just_finalized"] = None
        st["checks"] += 1                       # ★ گام ۶: شمارشِ بررسی‌ها

        # ★ گام ۵ — رأیِ بازبینی برای فردی که قبلاً سبز قفل شده بود.
        if st["locked"]:
            st["locked_frame"] = frame_idx      # ساعتِ بازبینی صفر شود
            if category == "green":
                return                          # هنوز صورتش باز است — کاری نکن
            # پوشش دیده شد → قفل می‌شکند و می‌رود به حالتِ فوکوس
            st["locked"] = False
            st["mode"] = "focus"
            st["votes"] = []
            st["focus_window"].clear()
            st["focus_window"].append(category)
            st["color"], st["label"] = category, LABELS[category]
            if category == "red":
                st["just_finalized"] = "red"
            return

        if st["mode"] == "fast":
            st["votes"].append((category, conf))
            if len(st["votes"]) >= self.FAST_VOTES_NEEDED:
                score = {"green": 0.0, "orange": 0.0, "red": 0.0}
                for c, cf in st["votes"]:
                    score[c] += cf
                best = max(score, key=score.get)
                # ★ A3 — سهمِ رأیِ برنده از کلِ امتیاز = ضریبِ اطمینان.
                #   قبلاً این عدد محاسبه می‌شد ولی دور ریخته می‌شد؛
                #   حالا هم روی تصویر نوشته و هم در JSON ثبت می‌شود.
                total = sum(score.values()) or 1.0

                # ★ D3 — قرمز باید *برتریِ روشن* داشته باشد، نه صرفاً
                #   بیشترین رأی. با ۲ نارنجی و ۱ قرمز، «قرمز» می‌توانست
                #   ببرد اگر اطمینانِ آن یک رأی بالا بود. برای آژیرِ
                #   امنیتی این کافی نیست: هزینهٔ آلارمِ کاذب بالاست.
                #   نارنجی چنین شرطی ندارد — سخت‌گیری فقط برای قرمز است.
                if (best == "red" and score["orange"] > 0
                        and score["red"] < self.RED_MARGIN * score["orange"]):
                    best = "orange"
                st["conf"] = score[best] / total
                if best == "green":
                    self._lock(tid, "green", frame_idx)
                else:
                    st["mode"] = "focus"
                    st["color"], st["label"] = best, LABELS[best]
                    st["focus_window"].append(best)
                    if best == "red":
                        st["just_finalized"] = "red"
        else:
            st["focus_window"].append(category)
            window = list(st["focus_window"])
            green_count = window.count("green")
            if green_count >= self.FOCUS_GREEN_NEEDED:
                self._lock(tid, "green", frame_idx)
            else:
                sub = [c for c in window if c in ("orange", "red")]
                if sub:
                    # ★ C3 — تساویِ آرا دیگر تصادفی نیست.
                    #   `max(set(sub), key=sub.count)` وقتی دو رنگ تعدادِ
                    #   برابر دارند، هرکدام را که در پیمایشِ set زودتر
                    #   بیاید برمی‌گرداند — و ترتیبِ set تضمین‌شده نیست.
                    #   یعنی با ۲ نارنجی و ۲ قرمز، رنگ می‌توانست بی‌دلیل
                    #   بین دو حالت بپرد. حالا در تساوی رنگِ فعلی حفظ
                    #   می‌شود؛ فقط با اکثریتِ واقعی عوض می‌شود.
                    counts = {c: sub.count(c) for c in set(sub)}
                    top = max(counts.values())
                    winners = [c for c, n in counts.items() if n == top]
                    if len(winners) > 1 and st["color"] in winners:
                        best = st["color"]           # تساوی → همان‌که هست
                    else:
                        best = sorted(winners)[0]    # قطعی و تکرارپذیر
                    # ★ D3 — در فوکوس هم قرمز اکثریتِ اکید لازم دارد.
                    #   به RED_MARGIN گره خورده تا وقتی آن را ۱.۰ می‌گذاری
                    #   (یعنی «سخت‌گیری خاموش»)، این هم خاموش شود. وگرنه
                    #   کلید نصفه‌نیمه عمل می‌کرد و گیج‌کننده می‌شد.
                    if (self.RED_MARGIN > 1.0 and best == "red"
                            and counts.get("red", 0) <= counts.get("orange", 0)):
                        best = "orange"
                    st["conf"] = counts[best] / len(window)   # ★ A3
                    if best == "red" and st["color"] != "red":
                        st["just_finalized"] = "red"
                    st["color"], st["label"] = best, LABELS[best]

    def _lock(self, tid, category, frame_idx=0):
        st = self.data[tid]
        st["locked"] = True
        st["locked_frame"] = frame_idx          # ★ گام ۵: شروعِ شمارشِ بازبینی
        st["color"] = category
        st["label"] = LABELS[category]

state_mgr = TrackStateManager()

---
## ۵ب) ★ C1 — حافظهٔ کوتاه‌مدتِ هویت

**مسئله:** فرد پشتِ قفسه می‌رود و برمی‌گردد → ByteTrack شناسهٔ تازه
می‌دهد → حکمِ قبلی از بین می‌رود. دزدِ قرمز بعد از دو ثانیه انسداد
دوباره «Analyzing» می‌شود.

**نگرانیِ درست: اگر دو نفر لباسِ شبیه داشته باشند چه؟** سه محافظ:

| محافظ | کار |
|---|---|
| حافظهٔ کوتاه | پیش‌فرض ۸ ثانیه. هرچه کوتاه‌تر، برخوردِ تصادفی کمتر |
| گیتِ مکانی | باید نزدیکِ محلِ ناپدیدشدن ظاهر شود و اندازهٔ جعبه هم‌خوان باشد |
| ★ آزمونِ حاشیه | اگر دو هویت **هر دو** شبیه باشند، هیچ‌کدام انتخاب نمی‌شود |

آزمونِ حاشیه مهم‌ترین است: در ابهام، سیستم ترجیح می‌دهد از صفر شروع
کند تا اینکه اشتباهی حکم را منتقل کند.

### و یک قانونِ ایمنی

> **هیچ‌وقت «قفل» به ارث نمی‌رسد.**

فردِ جدید رنگِ قبلی را نشان می‌دهد (پیوستگیِ بصری) ولی **دوباره
رأی‌گیری می‌شود**. اگر تطبیق اشتباه بوده باشد، ظرفِ چند فریم خودش را
اصلاح می‌کند.

اگر قفلِ سبز به ارث می‌رسید، یک تطبیقِ غلط می‌توانست یک دزد را برای
همیشه سبز کند — آن یک حفرهٔ امنیتی بود، نه یک اشکالِ کیفیت.

In [ ]:
# ---------------- 5b) ★ C1: حافظهٔ کوتاه‌مدتِ هویت --------------------
#
# مسئله: وقتی کسی پشتِ قفسه می‌رود و برمی‌گردد، ByteTrack شناسهٔ تازه
# می‌دهد. با شناسهٔ تازه، حکمِ قبلی از بین می‌رود و همه‌چیز از صفر
# شروع می‌شود — یعنی یک دزدِ قرمز بعد از یک انسدادِ دو ثانیه‌ای دوباره
# «Analyzing» می‌شود.
#
# سه محافظ در برابرِ اشتباه‌گرفتنِ افراد (مثلاً لباس‌های شبیه):
#
#   ۱) حافظهٔ کوتاه — پیش‌فرض ۸ ثانیه. هرچه بازه کوتاه‌تر، احتمالِ
#      برخوردِ تصادفیِ دو نفرِ شبیه کمتر.
#   ۲) گیتِ مکانی — فردِ جدید باید نزدیکِ جایی ظاهر شود که فردِ قبلی
#      ناپدید شده بود، و اندازهٔ جعبه‌اش هم هم‌خوان باشد.
#   ۳) ★ آزمونِ حاشیه — اگر دو هویتِ به‌یادمانده *هر دو* شبیه باشند
#      (مثلاً دو نفر با لباسِ هم‌رنگ)، هیچ‌کدام انتخاب نمی‌شود.
#      این مهم‌ترین محافظ است: در ابهام، سیستم ترجیح می‌دهد از صفر
#      شروع کند تا اینکه اشتباهی حکم را منتقل کند.
#
# و مهم‌تر از همه — قانونِ ایمنی:
#
#   ★ هیچ‌وقت «قفل» به ارث نمی‌رسد.
#     فردِ جدید رنگِ قبلی را نمایش می‌دهد (پیوستگیِ بصری) ولی
#     دوباره رأی‌گیری می‌شود. اگر تطبیق اشتباه بوده باشد، ظرفِ چند
#     فریم خودش را اصلاح می‌کند. اگر قفلِ سبز به ارث می‌رسید، یک
#     تطبیقِ غلط می‌توانست یک دزد را برای همیشه سبز کند.


class TrackMemory:
    def __init__(self, memory_seconds=8.0, fps=30.0,
                 match_th=0.80, margin=0.06, max_center_dist=0.35,
                 sig_every=5):
        self.ttl = int(memory_seconds * fps)
        self.match_th = match_th          # کمینهٔ شباهت برای پذیرش
        self.margin = margin              # فاصلهٔ لازم از دومین گزینه
        self.max_center_dist = max_center_dist   # نسبت به عرضِ فریم
        self.sig_every = sig_every
        self.slots = {}                   # tid -> {sig, box, frame, snap}
        self.stats = {"inherited": 0, "rejected_margin": 0, "rejected_far": 0}

    # ------------------------------------------------------------------
    @staticmethod
    def signature(crop):
        """امضای رنگیِ بدن — هیستوگرامِ Hue/Saturation. ارزان و ساده."""
        if crop is None or crop.size == 0:
            return None
        small = cv2.resize(crop, (48, 96), interpolation=cv2.INTER_AREA)
        hsv = cv2.cvtColor(small, cv2.COLOR_BGR2HSV)
        hist = cv2.calcHist([hsv], [0, 1], None, [24, 24], [0, 180, 0, 256])
        cv2.normalize(hist, hist, 0, 1, cv2.NORM_MINMAX)
        return hist.flatten()

    # ------------------------------------------------------------------
    def observe(self, tid, crop, box, frame_idx, st):
        """هر چند فریم یک بار، امضا و آخرین وضعیتِ فرد را به‌روز کن."""
        slot = self.slots.get(tid)
        need_sig = slot is None or (frame_idx - slot["frame"]) >= self.sig_every
        sig = self.signature(crop) if need_sig else slot["sig"]
        if sig is None:
            return
        self.slots[tid] = {
            "sig": sig, "box": tuple(float(v) for v in box), "frame": frame_idx,
            "snap": {"color": st["color"], "label": st["label"],
                     "mode": st["mode"], "conf": st.get("conf", 0.0)},
        }

    # ------------------------------------------------------------------
    def forget_old(self, frame_idx):
        dead = [t for t, s in self.slots.items()
                if (frame_idx - s["frame"]) > self.ttl]
        for t in dead:
            self.slots.pop(t, None)

    # ------------------------------------------------------------------
    def try_inherit(self, tid, st, crop, box, frame_idx, active_ids, frame_w):
        """
        اگر این شناسهٔ تازه به یکی از هویت‌های گم‌شده بخورد، رنگ و حالتش
        را به ارث می‌برد — ولی **قفل را نه**.
        """
        sig = self.signature(crop)
        if sig is None:
            return None

        cx = (box[0] + box[2]) / 2.0
        cy = (box[1] + box[3]) / 2.0
        bw = max(1.0, box[2] - box[0])
        limit = self.max_center_dist * frame_w

        scored = []
        for old_tid, slot in self.slots.items():
            if old_tid == tid or old_tid in active_ids:
                continue                                   # هنوز خودش فعال است
            if (frame_idx - slot["frame"]) > self.ttl:
                continue                                   # فراموش شده
            ob = slot["box"]
            ocx, ocy = (ob[0] + ob[2]) / 2.0, (ob[1] + ob[3]) / 2.0
            obw = max(1.0, ob[2] - ob[0])
            # ۲) گیتِ مکانی: هم فاصله، هم هم‌خوانیِ اندازه
            if ((cx - ocx) ** 2 + (cy - ocy) ** 2) ** 0.5 > limit:
                self.stats["rejected_far"] += 1
                continue
            if not (0.6 <= bw / obw <= 1.7):
                self.stats["rejected_far"] += 1
                continue
            score = float(cv2.compareHist(sig.reshape(24, 24),
                                          slot["sig"].reshape(24, 24),
                                          cv2.HISTCMP_CORREL))
            scored.append((score, old_tid))

        if not scored:
            return None
        scored.sort(reverse=True)
        best_score, best_tid = scored[0]
        if best_score < self.match_th:
            return None
        # ۳) آزمونِ حاشیه — اگر دومی هم تقریباً به همان خوبی است، رد کن
        if len(scored) > 1 and (best_score - scored[1][0]) < self.margin:
            self.stats["rejected_margin"] += 1
            return None

        snap = self.slots[best_tid]["snap"]
        # ★ قانونِ ایمنی: رنگ و حالت به ارث می‌رسد، قفل نه.
        st["color"] = snap["color"]
        st["label"] = snap["label"]
        st["conf"] = snap["conf"]
        st["locked"] = False
        st["mode"] = "focus" if snap["color"] in ("orange", "red") else "fast"
        st["votes"] = []
        self.slots.pop(best_tid, None)
        self.stats["inherited"] += 1
        return best_tid, best_score

---
## ۶) ابزارِ نمایش — گالری و اسکلت  ★ گام ۷

**★ گام ۷:** رنگِ پیش‌فرضِ `draw_upper_skeleton` دیگر زردِ ثابت نیست.
حالا فراخوان رنگِ وضعیتِ همان فرد را می‌فرستد، پس کادر و اسکلت و
برچسب هم‌رنگ‌اند.

In [ ]:
# ---------------- 6) Presentation Helpers (Gallery + Skeleton) --------
class PresentationGallery:
    """گالری تصاویر کوچک در گوشه تصویر - آخرین افراد شناسایی‌شده"""
    def __init__(self, max_items=4, thumb_size=140):
        self.items = deque(maxlen=max_items)   # هر آیتم: (img, label, color)
        self.thumb_size = thumb_size

    def add(self, crop_bgr, label, color):
        if crop_bgr is None or crop_bgr.size == 0:
            return
        thumb = cv2.resize(crop_bgr, (self.thumb_size, self.thumb_size))
        self.items.append((thumb, label, color))

    def draw(self, frame):
        h, w = frame.shape[:2]
        pad = 10
        for i, (thumb, label, color) in enumerate(self.items):
            x2 = w - pad
            x1 = x2 - self.thumb_size
            y1 = pad + i * (self.thumb_size + 35)
            y2 = y1 + self.thumb_size
            if y2 > h:
                break
            frame[y1:y2, x1:x2] = thumb
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 3)
            cv2.putText(frame, label, (x1, y2 + 20),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

gallery = PresentationGallery(max_items=4, thumb_size=140)

def draw_upper_skeleton(frame, kxy, kconf, color=(160, 160, 160), conf_th=0.4):
    """
    رسم اسکلتِ بالاتنه.

    ★ گام ۷ — رنگِ پیش‌فرض دیگر زردِ ثابت نیست. حالا فراخوان رنگِ
    وضعیتِ همان فرد را می‌فرستد، پس کادر و اسکلت و برچسب هم‌رنگ‌اند
    و کلِ فریم با یک نگاه خوانده می‌شود.
    """
    for a, b in UPPER_BODY_SKELETON:
        if kconf[a] < conf_th or kconf[b] < conf_th:
            continue
        pa = tuple(map(int, kxy[a]))
        pb = tuple(map(int, kxy[b]))
        cv2.line(frame, pa, pb, color, 2)
    for idx in [NOSE, LEYE, REYE, LEAR, REAR, LSHOULDER, RSHOULDER, LELBOW, RELBOW, LWRIST, RWRIST]:
        if kconf[idx] >= conf_th:
            p = tuple(map(int, kxy[idx]))
            cv2.circle(frame, p, 4, color, -1)

---
## ۷) خط لولهٔ اصلی

دو تابعِ تکراریِ نسخهٔ اصلی یکی شده‌اند. بررسی کردم — منطقِ تصمیم در
هر دو **یکسان** بود و نسخهٔ دمو فقط سه چیزِ نمایشی اضافه داشت:

| پرچم | پیش‌فرض | اثر |
|---|---|---|
| `show_skeleton` | `True` | رسمِ اسکلت (حالا هم‌رنگِ وضعیت) |
| `show_gallery` | `True` | گالریِ گوشهٔ تصویر |
| `slowmo_repeat` | `6` | تکرارِ فریم در لحظاتِ کلیدی (`1` = خاموش) |
| `min_eye_dist` | `8` | ★ گام ۶ |

**★ گام ۷ — ترتیبِ رسم عوض شد.** در نسخهٔ اصلی اسکلت در حلقهٔ *اول*
کشیده می‌شد، جایی که هنوز رنگِ وضعیت معلوم نبود. حالا کی‌پوینت‌ها در
`kpts_by_tid` نگه داشته می‌شوند و اسکلت در حلقهٔ *رسم* — بعد از
مشخص‌شدنِ رنگ — کشیده می‌شود.

**★ گام ۶ — برچسبِ پیشرفت.** در حالتِ خاکستری به‌جای
`Analyzing...` خالی، حالا `Analyzing... (2/3)` نوشته می‌شود؛ و اگر
هنوز هیچ بررسی‌ای ممکن نبوده `Analyzing... (too far)`. این‌طور معلوم
است سیستم فرد را دیده و دارد رویش کار می‌کند.

In [ ]:
# ---------------- 7) Main Pipeline ------------------------------------
def process_video(input_path, output_path, conf_thres=0.4, yolo_imgsz=640,
                  face_conf_th=0.5, slowmo_repeat=6,
                  show_skeleton=True, show_gallery=True, min_eye_dist=8,
                  track_memory=None, use_orientation=True,
                  tracker_cfg="bytetrack.yaml"):
    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        print("❌ Error: Cannot open video")
        return
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()

    out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))
    if not out.isOpened():                                        # [ایمنی]
        raise RuntimeError(f"❌ فایل خروجی باز نشد: {output_path}")
    t0 = time.time()
    last_pct = -1
    frame_idx = 0
    events = []                      # ★ A3: لاگِ رویدادها
    n_back = n_forced = 0            # ★ فاز E
    seen_ids = set()   # برای تشخیص "فرد کاملا جدید" جهت افکت اسلوموشن

    results_gen = pose_model.track(
        source=input_path, classes=[0], conf=conf_thres, imgsz=yolo_imgsz,
        tracker=tracker_cfg, stream=True, verbose=False, persist=True,
        half=USE_HALF
    )

    for r in results_gen:
        frame_idx += 1
        frame = r.orig_img

        if r.boxes.id is None or r.keypoints is None:
            out.write(frame)
            pct = int(frame_idx / total * 100) if total > 0 else 0
            if pct != last_pct and pct % 5 == 0:
                print(f"⏳ Progress: {pct}%"); last_pct = pct
            continue

        ids = r.boxes.id.int().cpu().tolist()
        boxes = r.boxes.xyxy.cpu().numpy()
        kpts_xy_all = r.keypoints.xy.cpu().numpy()
        kpts_conf_all = r.keypoints.conf.cpu().numpy() if r.keypoints.conf is not None else np.ones(kpts_xy_all.shape[:2])

        batch_crops, batch_meta = [], []
        trigger_slowmo = False   # آیا این فریم باید کند نمایش داده بشه؟
        kpts_by_tid = {}         # ★ گام ۷: برای رسمِ اسکلت با رنگِ وضعیت
        if track_memory is not None and frame_idx % 30 == 0:
            track_memory.forget_old(frame_idx)   # ★ C1: فراموشیِ دوره‌ای

        for tid, box, kxy, kconf in zip(ids, boxes, kpts_xy_all, kpts_conf_all):
            x1, y1, x2, y2 = [int(v) for v in box]
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(W, x2), min(H, y2)
            person_crop = frame[y1:y2, x1:x2]
            if person_crop.size == 0:
                continue

            # ★ C1 — اگر شناسه تازه است، شاید همان کسی باشد که چند
            #   ثانیه پیش پشتِ قفسه گمش کردیم.
            is_new_track = tid not in state_mgr.data
            st = state_mgr.ensure(tid)
            if track_memory is not None:
                if is_new_track:
                    track_memory.try_inherit(tid, st, person_crop, box,
                                             frame_idx, ids, W)
                track_memory.observe(tid, person_crop, box, frame_idx, st)

            kpts_by_tid[tid] = (kxy, kconf)          # ★ گام ۷

            # ★ فاز E — جهتِ سر (~۲۹ میکروثانیه)
            if use_orientation:
                _fa, _oc, _info = head_orientation(kxy, kconf, 0.35)
                st["is_back"], st["back_reason"] = \
                    back_verdict(st, _fa, _oc, _info)

                # ★ H5 — شیرِ اطمینان. حتی اگر «پشت» تشخیص
                #   داده شد، هر N فریم یک بار به‌هرحال بررسی کن —
                #   تا یک خطای پایدار نتواند کسی را برای همیشه
                #   پنهان کند. هزینه ناچیز، ولی خیال راحت.
                if st["is_back"]:
                    if (frame_idx - st["last_forced"]) >= ORI_SAFETY_VALVE:
                        st["last_forced"] = frame_idx
                        st["is_back"] = False
                        st["forced"] += 1
                        n_forced += 1

                if st["is_back"]:
                    st["ori_hist"]["back"] += 1
                elif _oc < ORI_MIN_CONF:
                    st["ori_hist"]["unknown"] += 1
                elif _fa >= 0.55:
                    st["ori_hist"]["front"] += 1
                else:
                    st["ori_hist"]["profile"] += 1
            else:
                st["is_back"] = False

            st["seen"] += 1
            if st["is_back"]:
                n_back += 1
                st["skipped_back"] += 1

            # --- افکت نمایشی: فرد کاملا جدید -> اسلوموشن + اضافه به گالری ---
            if tid not in seen_ids:
                seen_ids.add(tid)
                trigger_slowmo = True
                square_crop = person_crop.copy()
                gallery.add(square_crop, f"New Person ID {tid}", (255, 255, 0))

            # ★ فاز E — پشتش به دوربین است؟ قضاوت نکن.
            #   حکمِ قبلی دست‌نخورده می‌ماند — همان آدم است.
            if not st["is_back"] and state_mgr.should_analyze(tid, frame_idx):
                local_kxy = kxy.copy()
                local_kxy[:, 0] -= x1
                local_kxy[:, 1] -= y1

                face_crop, face_score = align_and_crop_face(
                    person_crop, local_kxy, kconf, face_conf_th, min_eye_dist)

                if face_crop is None:
                    st["skipped_small"] = st.get("skipped_small", 0) + 1
                if face_crop is not None:
                    batch_crops.append(face_crop)
                    batch_meta.append(tid)

        if batch_crops:
            results = classify_mask_batch(batch_crops)
            for (tid, (mask_label, conf), face_bgr) in zip(batch_meta, results, batch_crops):
                if mask_label == "no_mask":
                    state_mgr.register_vote(tid, "green", conf, frame_idx)
                else:
                    cat = "red" if is_suspicious(face_bgr) else "orange"
                    state_mgr.register_vote(tid, cat, conf, frame_idx)

        # ---- ★ A2: برشِ سوژه، از فریمی که هنوز رویش رسم نشده ----
        for tid, box in zip(ids, boxes):
            st = state_mgr.ensure(tid)
            if st.get("just_finalized") == "red":
                # ★ D4 — یک نفر نباید پشتِ سرِ هم آلارم بدهد.
                #   اگر وضعیتش بین نارنجی و قرمز نوسان کند، بدونِ
                #   این شرط هر نوسان یک «SUSPECT» تازه در گالری و
                #   یک رویدادِ تازه در JSON می‌ساخت — که دقیقاً مثلِ
                #   چند آلارمِ کاذبِ پشت‌سرهم دیده می‌شد.
                st["just_finalized"] = None
                if (frame_idx - st["last_alert"]) < state_mgr.ALERT_COOLDOWN:
                    continue
                st["last_alert"] = frame_idx
                trigger_slowmo = True
                x1c, y1c = max(0, int(box[0])), max(0, int(box[1]))
                x2c, y2c = min(W, int(box[2])), min(H, int(box[3]))
                if x2c > x1c and y2c > y1c:
                    gallery.add(frame[y1c:y2c, x1c:x2c].copy(),
                                f"SUSPECT ID {tid}", (0, 0, 255))
                events.append({                       # ★ A3
                    "frame": frame_idx,
                    "time_s": round(frame_idx / fps, 2),
                    "track_id": int(tid),
                    "state": "red",
                    "label": LABELS["red"],
                    "confidence": round(float(st.get("conf", 0.0)), 3),
                    "checks": int(st.get("checks", 0)),
                })

        # ---- رسم باکس‌ها ----
        for tid, box in zip(ids, boxes):
            x1, y1, x2, y2 = [int(v) for v in box]
            st = state_mgr.ensure(tid)
            color = COLORS[st["color"]]
            if not st["timeline"] or st["timeline"][-1][1] != st["color"]:
                st["timeline"].append((frame_idx, st["color"]))

            # ★ گام ۷ — اسکلت با رنگِ وضعیتِ همین فرد
            if show_skeleton and tid in kpts_by_tid:
                kxy_d, kconf_d = kpts_by_tid[tid]
                draw_upper_skeleton(frame, kxy_d, kconf_d, color)

            # ★ گام ۶ — در حالتِ «در حال بررسی» پیشرفت را نشان بده تا
            #   معلوم باشد سیستم فرد را دیده و دارد رویش کار می‌کند،
            #   نه اینکه او را نادیده گرفته باشد.
            # ★ A3 — ضریبِ اطمینان روی تصویر. تا حالا محاسبه می‌شد
            #   ولی هیچ‌جا دیده نمی‌شد؛ حالا هم روی کادر است و هم در JSON.
            label = st["label"]
            if st.get("is_back"):
                label = "Back to camera"          # ★ فاز E
            elif st["color"] != "gray" and st.get("conf", 0) > 0:
                label = f"{label} {st['conf']:.0%}"
            if st["color"] == "gray":
                got = len(st["votes"])
                if st["checks"] == 0:
                    label = "Analyzing... (too far)"
                else:
                    label = f"Analyzing... ({got}/{state_mgr.FAST_VOTES_NEEDED})"

            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame, f"ID {tid}: {label}", (x1, max(20, y1 - 8)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)

            if st["color"] == "red":
                cv2.putText(frame, "ALERT!", (x1, y2 + 22),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)


        # ---- گالری گوشه تصویر ----
        if show_gallery:
            gallery.draw(frame)

        # ---- نوشتن خروجی (با افکت اسلوموشن در لحظات کلیدی) ----
        repeat = slowmo_repeat if trigger_slowmo else 1
        for _ in range(repeat):
            out.write(frame)

        pct = int(frame_idx / total * 100) if total > 0 else 0
        if pct != last_pct and pct % 5 == 0:
            print(f"⏳ Progress: {pct}%")
            last_pct = pct

    out.release()

    # ---- ★ A3: وضعیتِ نهاییِ هر فرد هم ثبت شود ----
    for tid, st in state_mgr.data.items():
        events.append({
            "frame": frame_idx,
            "time_s": round(frame_idx / fps, 2),
            "track_id": int(tid),
            "state": st["color"],
            "label": st["label"],
            "confidence": round(float(st.get("conf", 0.0)), 3),
            "checks": int(st.get("checks", 0)),
            "final": True,
        })

    json_path = os.path.splitext(output_path)[0] + "_events.json"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump({"video": input_path,
                   "frames": frame_idx,
                   "pose_weights": POSE_WEIGHTS,
                   "params": {"conf_thres": conf_thres,
                              "yolo_imgsz": yolo_imgsz,
                              "face_conf_th": face_conf_th,
                              "min_eye_dist": min_eye_dist},
                   "events": events}, f, ensure_ascii=False, indent=2)

    print(f"✅ Done in {time.time() - t0:.2f}s -> {output_path}")
    print(f"📄 {len(events)} رویداد -> {json_path}")
    if use_orientation:
        print(f"🔄 فاز E: {n_back} فریم «پشت» (قضاوت نشد) | "
              f"{n_forced} بررسیِ اجباری (شیرِ اطمینان)")
    return events

---
## ۸) اجرا

### اتصال Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### تنظیم مسیرها و اجرا

In [ ]:
import os

if not os.path.exists(INPUT_VIDEO):                               # [ایمنی]
    raise FileNotFoundError(f"❌ ویدیو پیدا نشد: {INPUT_VIDEO}")

# ── ★ فایلِ پیکربندیِ ردیاب ─────────────────────────────────────────
#   اعداد از سلولِ تنظیماتِ بالا می‌آیند و آنجا تک‌تک توضیح داده شده‌اند.
#   خلاصه‌اش: تلاشِ اولیه برای پایدارکردنِ شناسه با بالابردنِ
#   `track_buffer` و `match_thresh` نتیجهٔ معکوس داد — مسیرِ فردِ
#   گم‌شده روی نفرِ کناری سُر می‌خورد و حکمش را به او منتقل می‌کرد.
#   راهِ درست عکسِ آن بود: تطبیقِ سخت‌گیر (`match_thresh=0.60`)، بافرِ
#   کوتاه (۳۰) و زنده‌کردنِ مرحلهٔ دومِ ByteTrack با پایین‌آوردنِ
#   `CONF_THRES` تا هم‌ترازِ `track_low_thresh`. هزینهٔ محاسباتی: صفر.
TRACKER_CFG = "tracker_custom.yaml"
with open(TRACKER_CFG, "w") as f:
    f.write(f"""tracker_type: bytetrack
track_high_thresh: {TRACK_HIGH_THRESH}
track_low_thresh: {TRACK_LOW_THRESH}
new_track_thresh: {NEW_TRACK_THRESH}
track_buffer: {TRACK_BUFFER}
match_thresh: {MATCH_THRESH}
fuse_score: true
""")
print(f"🎯 ردیاب: buffer={TRACK_BUFFER} فریم  new_thresh={NEW_TRACK_THRESH}")

# ── صفرکردنِ حالتِ داخلی ────────────────────────────────────────────
#   بدونِ این، اگر سلول را دو بار اجرا کنی شناسه‌ها و رأی‌های اجرای
#   قبلی باقی می‌مانند و نتیجه اشتباه می‌شود.
state_mgr = TrackStateManager()
gallery   = PresentationGallery(max_items=4, thumb_size=140)
track_memory = (TrackMemory(memory_seconds=MEMORY_SECONDS, fps=30.0,
                            match_th=REID_MATCH_TH)
                if USE_TRACK_MEMORY else None)

events = process_video(INPUT_VIDEO, OUTPUT_VIDEO,
                       conf_thres=CONF_THRES,
                       yolo_imgsz=YOLO_IMGSZ,
                       face_conf_th=FACE_CONF_TH,
                       slowmo_repeat=SLOWMO_REPEAT,
                       show_skeleton=SHOW_SKELETON,
                       show_gallery=SHOW_GALLERY,
                       min_eye_dist=MIN_EYE_DIST,
                       track_memory=track_memory,
                       use_orientation=USE_ORIENTATION,
                       tracker_cfg=TRACKER_CFG)

### نمایش ویدیوی خروجی

In [ ]:
import subprocess, os
from base64 import b64encode
from IPython.display import HTML

web = "/content/preview.mp4"
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", OUTPUT_VIDEO,
                "-vcodec", "libx264", "-crf", "26", web], check=False)

path = web if os.path.exists(web) else OUTPUT_VIDEO
data = b64encode(open(path, "rb").read()).decode()
HTML(f'<video width=860 controls>'
     f'<source src="data:video/mp4;base64,{data}" type="video/mp4"></video>')

### گزارشِ وضعیتِ نهاییِ هر فرد

بعد از اجرا، این سلول می‌گوید سیستم دربارهٔ هر نفر به چه نتیجه‌ای
رسید و چند بار بررسی‌اش کرد — برای راستی‌آزمایی مفید است.

In [ ]:
from collections import Counter

rows = []
for tid, st in sorted(state_mgr.data.items()):
    rows.append((tid, st["color"], st["checks"], st["locked"], st["mode"]))

print(f"{'ID':>4}  {'وضعیت':<8} {'بررسی':>6} {'قفل':>5}  حالت")
print("-" * 44)
for tid, color, checks, locked, mode in rows:
    print(f"{tid:>4}  {color:<8} {checks:>6} {str(locked):>5}  {mode}")

print("\nجمع‌بندی:", dict(Counter(r[1] for r in rows)))
print("مجموع بررسی‌ها:", sum(r[2] for r in rows))

---
## ★ گزارشِ دقیقِ هر شناسه

مهم‌ترین سلول برای **راستی‌آزماییِ الگوریتم**.

| ستون | معنی |
|---|---|
| `frames` | چند فریم در کادر بود |
| `front / prof / unk / back` | جهتِ سر در آن فریم‌ها |
| `checks` | چند بار واقعاً به طبقه‌بند رفت |
| `skipB` | چند بار **به‌خاطرِ پشت‌بودن** قضاوت نشد |
| `force` | چند بار شیرِ اطمینان بررسیِ اجباری کرد |
| `from` | هویتش از کدام شناسه به ارث رسید |

**چه چیزی را ببین:**

- `prof > 0` با `checks > 0` یعنی نیم‌رخ‌ها **رد نشده‌اند** ✔
- `back > 0` یعنی فاز E فعال شده. `back = 0` یعنی یا کسی پشتش به
  دوربین نبوده یا آستانه سخت‌گیر است.
- `from` یعنی حافظهٔ هویت همان آدم را دنبال کرده، نه اینکه شناسهٔ
  تازه ساخته باشد.
- اگر کسی `back` زیاد دارد ولی در ویدیو رو به دوربین بوده، دلیلش را
  در بخشِ «چرا پشت» ببین و `ORI_BACK_ENTER` را منفی‌تر کن.

In [ ]:
S = state_mgr.data
if not S:
    print("هیچ شناسه‌ای ثبت نشده — اول سلولِ اجرا را ران کن.")
else:
    FA = {"gray": "خاکستری", "green": "سبز",
          "orange": "نارنجی", "red": "🔴 قرمز"}

    hdr = (f"{'ID':>4} {'frames':>7} {'front':>6} {'prof':>5} {'unk':>4} "
           f"{'back':>5} {'checks':>7} {'skipB':>6} {'force':>6} "
           f"{'from':>5}  verdict")
    print(hdr); print("-" * len(hdr))
    for tid in sorted(S):
        st = S[tid]; o = st["ori_hist"]; frm = st["inherited_from"]
        conf = f"{st['conf']:.0%}" if st["conf"] > 0 else "-"
        print(f"{tid:>4} {st['seen']:>7} {o['front']:>6} {o['profile']:>5} "
              f"{o['unknown']:>4} {o['back']:>5} {st['checks']:>7} "
              f"{st['skipped_back']:>6} {st['forced']:>6} "
              f"{(str(frm) if frm else '-'):>5}  "
              f"{FA.get(st['color'], st['color'])} {conf}")

    tb = sum(s["ori_hist"]["back"] for s in S.values())
    tp = sum(s["ori_hist"]["profile"] for s in S.values())
    tc = sum(s["checks"] for s in S.values())
    inh = [t for t, s in S.items() if s["inherited_from"]]

    print("\n" + "=" * 62)
    print(f"  شناسه‌ها        : {len(S)}")
    print(f"  کلِ بررسی‌ها    : {tc}")
    print(f"  فریم‌های نیم‌رخ : {tp}   ← این‌ها تحلیل شده‌اند ✔")
    print(f"  فریم‌های پشت    : {tb}   ← این‌ها قضاوت نشده‌اند")
    print(f"  وراثتِ هویت     : {len(inh)} شناسه {inh if inh else ''}")

    # ---- چرا «پشت»؟ ----
    backs = [(t, s) for t, s in S.items() if s["ori_hist"]["back"] > 0]
    if backs:
        print("\n" + "=" * 62)
        print("  چرا «پشت به دوربین» اعلام شد")
        print("=" * 62)
        for t, s in backs:
            print(f"  #{t:<4} {s['back_reason']}")
    else:
        print("\n  ℹ️ هیچ فریمی «پشت» تشخیص داده نشد.")
        print("     برای حساس‌ترکردن: ORI_BACK_ENTER = -0.25")

    # ---- خطِ زمانی ----
    print("\n" + "=" * 62)
    print("  خطِ زمانیِ حکم (فقط لحظاتِ تغییر)")
    print("=" * 62)
    for tid in sorted(S):
        tl = S[tid]["timeline"]
        if not tl: continue
        path = " → ".join(f"f{f}:{FA.get(c, c)}" for f, c in tl[:8])
        more = f"  (+{len(tl)-8})" if len(tl) > 8 else ""
        print(f"  #{tid:<4} {path}{more}")

### ذخیره در Google Drive

In [ ]:
import shutil, os

# DRIVE_DEST از سلولِ تنظیماتِ بالا می‌آید
if os.path.exists(OUTPUT_VIDEO):
    shutil.copy(OUTPUT_VIDEO, DRIVE_DEST)
    print(f"✅ ذخیره شد: {DRIVE_DEST}")
else:
    print("❌ فایل خروجی پیدا نشد — اول سلولِ اجرا را ران کن.")